In [1]:
# Load the extension that allows us to compile CUDA code in python notebooks
# Documentation is here: https://nvcc4jupyter.readthedocs.io/en/latest/
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-zr2jwr8b
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-zr2jwr8b
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 173e636d9a3cb81c6101a160c1c8c86864eaaf5b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nvcc4jupyter: filename=nvcc4jupyter-1.2.2-py3-none-any.whl size=12545 sha256=f293982cdc354d66d92473c4350316b6ff6623d7028219d8cf93377ffb402266
  Stored in directory: /tmp/pip-ephem-wheel-cache-qrx48k3s/wheels/2f/1e/79/584a8d85ad018ecc510baa90d4fbc161bd56a4d66943d5b0d4
Successfully built nvcc4jupyter
Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpm4ityugb".


In [2]:
%%cuda_group_save -g "source" -n "data_types.h"
/**
 * A collection of commonly used data types throughout this project.
 */
#pragma once

#include <iostream> // for std::ostream
#include <vector>

namespace csc485b{
namespace a2{

using node_t = int;
using edge_t = int2;

using edge_list_t = std::vector< edge_t >;
using node_list_t = std::vector< node_t >;

} // namespace a2
} // namespace csc485b


In [3]:
%%cuda_group_save -g "source" -n "cuda_common.h"
/**
 * Standard macros that can be useful for error checking.
 * https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__ERROR.html
 */
#pragma once

#include <cuda.h>

#define CUDA_CALL(exp)                                       \
    do {                                                     \
        cudaError res = (exp);                               \
        if(res != cudaSuccess) {                             \
            printf("Error at %s:%d\n %s\n",                  \
                __FILE__,__LINE__, cudaGetErrorString(res)); \
           exit(EXIT_FAILURE);                               \
        }                                                    \
    } while(0)

#define CHECK_ERROR(msg)                                             \
    do {                                                             \
        cudaError_t err = cudaGetLastError();                        \
        if(cudaSuccess != err) {                                     \
            printf("Error (%s) at %s:%d\n %s\n",                     \
                (msg), __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                      \
        }                                                            \
    } while (0)

In [4]:
%%cuda_group_save -g "source" -n "data_generator.h"
/**
 * Functions for generating random input data with a fixed seed
 */
#pragma once

#include <cassert>  // for assert()
#include <cstddef>  // std::size_t type
#include <random>   // for std::mt19937, std::uniform_int_distribution
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a2 {

/**
 * Generates and returns a vector of random edges
 * for a graph `G=(V,E)` with `n=|V|=n` and expected `m=|E|`.
 * Referred to as an Erdős-Rényi graph.
 *
 * @see https://networkx.org/documentation/stable/reference/generated/networkx.generators.random_graphs.fast_gnp_random_graph.html#networkx.generators.random_graphs.fast_gnp_random_graph
 */
edge_list_t generate_graph( std::size_t n, std::size_t m )
{
    assert( "At most n(n-1) edges in a simple graph" && m < n * ( n - 1 ) );

    int const probability = ( 100 * m ) / ( n * ( n - 1 ) );

    // for details of random number generation, see:
    // https://en.cppreference.com/w/cpp/numeric/random/uniform_int_distribution
    std::size_t random_seed = 20241008;  // use magic seed
    std::mt19937 rng( random_seed );     // use mersenne twister generator
    std::uniform_int_distribution<> distrib(0, 100);

    edge_list_t random_edges;
    random_edges.reserve( 2 * m );

    for( node_t u = 0; u < n; ++u )
    {
        for( node_t v = u + 1; v < n; ++v )
        {
            auto const dice_roll = distrib( rng );
            if( dice_roll <= probability )
            {
                random_edges.push_back( make_int2( u, v ) );
                random_edges.push_back( make_int2( v, u ) );
            }
        }
    }

    random_edges.resize( random_edges.size() );


    return random_edges;
}

} // namespace a2
} // namespace csc485b


In [5]:
%%cuda_group_save -g "source" -n "dense_graph.h"
/**
 * The file in which you will implement your DenseGraph GPU solutions!
 */

#include <cstddef>  // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A DenseGraph is optimised for a graph in which the number of edges
 * is close to n(n-1). It is represented using an adjacency matrix.
 */
struct DenseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  node_t * adjacencyMatrix; /** Pointer to an n x n adj. matrix */

  /** Returns number of cells in the adjacency matrix. */
  __device__ __host__ __forceinline__
  std::size_t matrix_size() const { return n * n; }
};

namespace gpu {

inline constexpr int DENSE_TILE_SIZE = 16;

/**
 * Constructs a DenseGraph from an input edge list of m edges.
 *
 * @pre The pointers in DenseGraph g have already been allocated.
 */
__global__
void build_graph( DenseGraph g, edge_t const * edge_list, std::size_t m )
{
    //get the threadid
    int threadid = blockIdx.x * blockDim.x + threadIdx.x;

    //if the thread's id is bigger than the edge pair list we exit
    if(threadid >= (int)m){
        return;
    }

    //each thread grab a edge pair and add him to the adjacency matrix
    int row = edge_list[threadid].x;
    int column = edge_list[threadid].y;

    //put a 1 at row x column y
    //edge_list already has both directions from the data generator
    g.adjacencyMatrix[row * g.n + column] = 1;
}

/**
 * Repopulates the adjacency matrix as a new graph that represents
 * the two-hop neighbourhood of input graph g
 */
__global__
void two_hop_reachability( DenseGraph g, node_t * output )
{
    //load tiles into shared memory so we dont keep reading the same values
    __shared__ node_t row_tile[DENSE_TILE_SIZE][DENSE_TILE_SIZE];
    __shared__ node_t column_tile[DENSE_TILE_SIZE][DENSE_TILE_SIZE];

    int row = blockIdx.y * DENSE_TILE_SIZE + threadIdx.y;
    int column = blockIdx.x * DENSE_TILE_SIZE + threadIdx.x;
    int result = 0;

    for(int tile_start = 0; tile_start < (int)g.n; tile_start += DENSE_TILE_SIZE){
        int row_column = tile_start + threadIdx.x;
        int column_row = tile_start + threadIdx.y;

        row_tile[threadIdx.y][threadIdx.x] = (row < (int)g.n && row_column < (int)g.n) ? g.adjacencyMatrix[row * g.n + row_column] : 0;
        column_tile[threadIdx.y][threadIdx.x] = (column < (int)g.n && column_row < (int)g.n) ? g.adjacencyMatrix[column_row * g.n + column] : 0;

        __syncthreads();

        if(row < (int)g.n && column < (int)g.n && row != column){
            for(int current = 0; current < DENSE_TILE_SIZE; current++){
                result += row_tile[threadIdx.y][current] * column_tile[current][threadIdx.x];
            }
        }

        __syncthreads();
    }

    if(row < (int)g.n && column < (int)g.n){
        output[row * g.n + column] = (row != column && result > 0) ? 1 : 0;
    }
}

} // namespace gpu
} // namespace a2
} // namespace csc485b


In [6]:
%%cuda_group_save -g "source" -n "sparse_graph.h"
/**
 * The file in which you will implement your SparseGraph GPU solutions!
 */

#include <cstddef>  // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A SparseGraph is optimised for a graph in which the number of edges
 * is close to cn, for a small constanct c. It is represented in CSR format.
 */
struct SparseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  std::size_t m; /**< Number of edges in the graph. */
  node_t * neighbours_start_at; /** Pointer to an n=|V| offset array */
  node_t * neighbours; /** Pointer to an m=|E| array of edge destinations */
};

namespace gpu {

inline constexpr int SPARSE_THREADS = 256;

/**
 * Constructs a SparseGraph from an input edge list of m edges.
 *
 * @pre The pointers in SparseGraph g have already been allocated.
 */
__global__
void build_graph( SparseGraph g, edge_t const * edge_list, std::size_t m )
{
    //get the threadid
    int threadid = blockIdx.x * blockDim.x + threadIdx.x;

    //if the thread's id is bigger than the edge pair list we exit
    if(threadid >= (int)m){
        return;
    }

    int row = edge_list[threadid].x;

    //count the number of neighbours in each row
    atomicAdd(&g.neighbours_start_at[row], 1);
}

__global__
void make_offsets( SparseGraph g, node_t const * counts )
{
    int threadid = blockIdx.x * blockDim.x + threadIdx.x;

    if(threadid >= (int)g.n){
        return;
    }

    //find where this row starts
    node_t sum = 0;
    for(int i = 0; i < threadid; i++){
        sum += counts[i];
    }

    g.neighbours_start_at[threadid] = sum;
}

__global__
void make_neighbours( SparseGraph g, edge_t const * edge_list, std::size_t m, node_t * next_spot )
{
    int threadid = blockIdx.x * blockDim.x + threadIdx.x;

    if(threadid >= (int)m){
        return;
    }

    int row = edge_list[threadid].x;
    int column = edge_list[threadid].y;

    //get the next empty spot in this row
    int spot = atomicAdd(&next_spot[row], 1);
    g.neighbours[spot] = column;
}

__global__
void sort_neighbours( SparseGraph g )
{
    int row = blockIdx.x * blockDim.x + threadIdx.x;

    if(row >= (int)g.n){
        return;
    }

    int start = g.neighbours_start_at[row];
    int end = (row + 1 < (int)g.n) ? g.neighbours_start_at[row + 1] : (int)g.m;

    //keep each row in order
    for(int i = start + 1; i < end; i++){
        node_t value = g.neighbours[i];
        int current = i - 1;

        while(current >= start && g.neighbours[current] > value){
            g.neighbours[current + 1] = g.neighbours[current];
            current--;
        }

        g.neighbours[current + 1] = value;
    }
}

/**
 * Repopulates the adjacency lists as a new graph that represents
 * the two-hop neighbourhood of input graph g
 */
__global__
void two_hop_reachability( SparseGraph g, unsigned char * reachable, node_t * output_counts )
{
    //one thread handles one starting node
    int row = blockIdx.x * blockDim.x + threadIdx.x;

    if(row >= (int)g.n){
        return;
    }

    int row_start = g.neighbours_start_at[row];
    int row_end = (row + 1 < (int)g.n) ? g.neighbours_start_at[row + 1] : (int)g.m;
    int count = 0;

    for(int edge = row_start; edge < row_end; edge++){
        int middle = g.neighbours[edge];
        int middle_start = g.neighbours_start_at[middle];
        int middle_end = (middle + 1 < (int)g.n) ? g.neighbours_start_at[middle + 1] : (int)g.m;

        //look at every neighbour of this middle node
        for(int next_edge = middle_start; next_edge < middle_end; next_edge++){
            int column = g.neighbours[next_edge];
            std::size_t spot = row * g.n + column;

            //dont add the start node or the same node twice
            if(column != row && reachable[spot] == 0){
                reachable[spot] = 1;
                count++;
            }
        }
    }

    output_counts[row] = count;
}

__global__
void make_reachability_neighbours( SparseGraph output, unsigned char const * reachable )
{
    int row = blockIdx.x * blockDim.x + threadIdx.x;

    if(row >= (int)output.n){
        return;
    }

    int spot = output.neighbours_start_at[row];

    for(int column = 0; column < (int)output.n; column++){
        if(reachable[row * output.n + column] != 0){
            output.neighbours[spot] = column;
            spot++;
        }
    }
}

} // namespace gpu
} // namespace a2
} // namespace csc485b


In [7]:
%%cuda_group_save -g "source" -n "main.cu"

#include <chrono>
#include <cstdlib>
#include <iostream>

#include "cuda_common.h"
#include "data_generator.h"
#include "data_types.h"
#include "dense_graph.h"
#include "sparse_graph.h"

//find how many blocks we need
int get_blocks(std::size_t amount, int threads)
{
    return (amount + threads - 1) / threads;
}

//allocate and run the dense graph
void run_dense(csc485b::a2::edge_t const * d_edges, std::size_t n, std::size_t m)
{
    using namespace csc485b;

    std::size_t const matrix_size = n * n;
    std::size_t const matrix_bytes = matrix_size * sizeof(a2::node_t);

    a2::node_t * d_matrix = nullptr;
    a2::node_t * d_output = nullptr;

    CUDA_CALL(cudaMalloc((void**)&d_matrix, matrix_bytes));
    CUDA_CALL(cudaMalloc((void**)&d_output, matrix_bytes));

    a2::DenseGraph graph{n, d_matrix};

    int const threads = 256;
    int const blocks = get_blocks(m, threads);

    CUDA_CALL(cudaDeviceSynchronize());
    auto const build_start = std::chrono::high_resolution_clock::now();

    CUDA_CALL(cudaMemset(d_matrix, 0, matrix_bytes));

    if(m > 0){
        a2::gpu::build_graph<<<blocks, threads>>>(graph, d_edges, m);
        CHECK_ERROR("dense build_graph");
    }

    CUDA_CALL(cudaDeviceSynchronize());
    auto const reachability_start = std::chrono::high_resolution_clock::now();

    int constexpr tile = a2::gpu::DENSE_TILE_SIZE;
    dim3 const matrix_threads(tile, tile);
    dim3 const matrix_blocks((n + tile - 1) / tile, (n + tile - 1) / tile);

    a2::gpu::two_hop_reachability<<<matrix_blocks, matrix_threads>>>(graph, d_output);
    CHECK_ERROR("dense two_hop_reachability");

    CUDA_CALL(cudaDeviceSynchronize());
    auto const reachability_end = std::chrono::high_resolution_clock::now();

    //put the result back into the graph
    CUDA_CALL(cudaMemcpy(d_matrix, d_output, matrix_bytes, cudaMemcpyDeviceToDevice));

    auto const build_time =
        std::chrono::duration_cast<std::chrono::microseconds>(
            reachability_start - build_start
        ).count();

    auto const reachability_time =
        std::chrono::duration_cast<std::chrono::microseconds>(
            reachability_end - reachability_start
        ).count();

    std::cout << "Dense build time: " << build_time << " us\n";
    std::cout << "Dense reachability time: " << reachability_time << " us\n";

    CUDA_CALL(cudaFree(d_output));
    CUDA_CALL(cudaFree(d_matrix));
}

//allocate and run the sparse graph
void run_sparse(csc485b::a2::edge_t const * d_edges, std::size_t n, std::size_t m)
{
    using namespace csc485b;

    std::size_t const node_bytes = n * sizeof(a2::node_t);
    std::size_t const neighbour_bytes = ((m > 0) ? m : 1) * sizeof(a2::node_t);
    std::size_t const matrix_size = n * n;

    a2::node_t * d_offsets = nullptr;
    a2::node_t * d_neighbours = nullptr;
    a2::node_t * d_counts = nullptr;
    a2::node_t * d_next_spot = nullptr;

    a2::node_t * d_output_offsets = nullptr;
    a2::node_t * d_output_neighbours = nullptr;
    a2::node_t * d_output_counts = nullptr;

    unsigned char * d_reachable = nullptr;

    CUDA_CALL(cudaMalloc((void**)&d_offsets, node_bytes));
    CUDA_CALL(cudaMalloc((void**)&d_neighbours, neighbour_bytes));
    CUDA_CALL(cudaMalloc((void**)&d_counts, node_bytes));
    CUDA_CALL(cudaMalloc((void**)&d_next_spot, node_bytes));

    CUDA_CALL(cudaMalloc((void**)&d_output_offsets, node_bytes));
    CUDA_CALL(cudaMalloc(
        (void**)&d_output_neighbours,
        matrix_size * sizeof(a2::node_t)
    ));
    CUDA_CALL(cudaMalloc((void**)&d_output_counts, node_bytes));
    CUDA_CALL(cudaMalloc(
        (void**)&d_reachable,
        matrix_size * sizeof(unsigned char)
    ));

    a2::SparseGraph graph{n, m, d_offsets, d_neighbours};
    a2::SparseGraph output_graph{
        n,
        0,
        d_output_offsets,
        d_output_neighbours
    };

    int const threads = a2::gpu::SPARSE_THREADS;
    int const edge_blocks = get_blocks(m, threads);
    int const node_blocks = get_blocks(n, threads);

    CUDA_CALL(cudaDeviceSynchronize());
    auto const build_start = std::chrono::high_resolution_clock::now();

    CUDA_CALL(cudaMemset(d_offsets, 0, node_bytes));

    if(m > 0){
        a2::gpu::build_graph<<<edge_blocks, threads>>>(graph, d_edges, m);
        CHECK_ERROR("sparse build_graph");
    }

    CUDA_CALL(cudaMemcpy(
        d_counts,
        d_offsets,
        node_bytes,
        cudaMemcpyDeviceToDevice
    ));

    a2::gpu::make_offsets<<<node_blocks, threads>>>(graph, d_counts);
    CHECK_ERROR("sparse make_offsets");

    CUDA_CALL(cudaMemcpy(
        d_next_spot,
        d_offsets,
        node_bytes,
        cudaMemcpyDeviceToDevice
    ));

    if(m > 0){
        a2::gpu::make_neighbours<<<edge_blocks, threads>>>(
            graph,
            d_edges,
            m,
            d_next_spot
        );
        CHECK_ERROR("sparse make_neighbours");
    }

    a2::gpu::sort_neighbours<<<node_blocks, threads>>>(graph);
    CHECK_ERROR("sparse sort_neighbours");

    CUDA_CALL(cudaDeviceSynchronize());
    auto const reachability_start = std::chrono::high_resolution_clock::now();

    CUDA_CALL(cudaMemset(
        d_reachable,
        0,
        matrix_size * sizeof(unsigned char)
    ));

    a2::gpu::two_hop_reachability<<<node_blocks, threads>>>(
        graph,
        d_reachable,
        d_output_counts
    );
    CHECK_ERROR("sparse two_hop_reachability");

    a2::gpu::make_offsets<<<node_blocks, threads>>>(
        output_graph,
        d_output_counts
    );
    CHECK_ERROR("sparse reachability offsets");

    a2::gpu::make_reachability_neighbours<<<node_blocks, threads>>>(
        output_graph,
        d_reachable
    );
    CHECK_ERROR("sparse reachability neighbours");

    CUDA_CALL(cudaDeviceSynchronize());

    a2::node_t last_offset = 0;
    a2::node_t last_count = 0;

    CUDA_CALL(cudaMemcpy(
        &last_offset,
        d_output_offsets + n - 1,
        sizeof(a2::node_t),
        cudaMemcpyDeviceToHost
    ));

    CUDA_CALL(cudaMemcpy(
        &last_count,
        d_output_counts + n - 1,
        sizeof(a2::node_t),
        cudaMemcpyDeviceToHost
    ));

    output_graph.m = last_offset + last_count;

    auto const reachability_end = std::chrono::high_resolution_clock::now();

    auto const build_time =
        std::chrono::duration_cast<std::chrono::microseconds>(
            reachability_start - build_start
        ).count();

    auto const reachability_time =
        std::chrono::duration_cast<std::chrono::microseconds>(
            reachability_end - reachability_start
        ).count();

    std::cout << "Sparse build time: " << build_time << " us\n";
    std::cout << "Sparse reachability time: " << reachability_time << " us\n";

    CUDA_CALL(cudaFree(d_reachable));
    CUDA_CALL(cudaFree(d_output_counts));
    CUDA_CALL(cudaFree(d_output_neighbours));
    CUDA_CALL(cudaFree(d_output_offsets));
    CUDA_CALL(cudaFree(d_next_spot));
    CUDA_CALL(cudaFree(d_counts));
    CUDA_CALL(cudaFree(d_neighbours));
    CUDA_CALL(cudaFree(d_offsets));
}

int main()
{
    using namespace csc485b;

    std::size_t const n = 4096;
    std::size_t const expected_m = n * 4;

    a2::edge_list_t edges = a2::generate_graph(n, expected_m);
    std::size_t const m = edges.size();
    a2::edge_t * d_edges = nullptr;

    if(m > 0){
        CUDA_CALL(cudaMalloc(
            (void**)&d_edges,
            m * sizeof(a2::edge_t)
        ));

        CUDA_CALL(cudaMemcpy(
            d_edges,
            edges.data(),
            m * sizeof(a2::edge_t),
            cudaMemcpyHostToDevice
        ));
    }

    run_dense(d_edges, n, m);
    run_sparse(d_edges, n, m);

    CUDA_CALL(cudaFree(d_edges));
    return EXIT_SUCCESS;
}

In [8]:
%cuda_group_run --group "source" --compiler-args "-O3 -g -std=c++20 -arch=sm_75"

Dense build time: 926 us
Dense reachability time: 199948 us
Sparse build time: 365 us
Sparse reachability time: 7059 us

